[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/humanization/humanization_guide/04_sequence_qc/04_numbering_lab.ipynb)


# 04 — 입력 QC — numbering · CDR · germline · 번호 체계

> 본문 [`04_sequence_qc.md`](04_sequence_qc.md) 와 **한 절씩 짝지어** 보세요.
> **실측 소요 —** ANARCI numbering+germline(H·L 동시) **0.15초** · abnumber CDR 추출 **1초 미만**

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `04_sequence_qc/` 로 이동해 필요한 패키지만 깝니다(로컬에서 `04_sequence_qc/` 안에 열었다면 클론 생략).
끝나면 **`my_run/`**(내가 만들 결과)과 **`data/`**(커밋된 레퍼런스)가 준비돼요 — 랩은 `my_run/` 을 먼저 찾고 없으면 `data/` 로 폴백하며 어느 쪽을 썼는지 매번 print 합니다.
- ANARCI 는 numbering 을 **hmmscan(HMMER)** 서브프로세스로 돌려요. Colab 에서는 `apt-get install -y hmmer` 가 함께 실행돼요 — pip 만으로는 `hmmscan` 이 없어 죽어요.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "04_sequence_qc"
APT_PKGS = "hmmer"     # Colab 에서만: 시스템 패키지 (hmmer = ANARCI 가 부르는 hmmscan)
PIP_PKGS = "pandas anarci abnumber"     # 없는 것만 설치

import os, sys, json, time, shutil, pathlib, subprocess, importlib, importlib.util
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막아요.
# (멈춤은 예외가 안 나서 폴백이 안 걸려요 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어져요)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, check=True, timeout=None, quiet=False):
    # timeout 을 꼭 주세요 — 네트워크가 '멈춘 채' 매달리면 예외가 안 나서 data/ 폴백이 안 걸립니다.
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        return subprocess.run(cmd, shell=True, check=check, timeout=timeout)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        if check:
            raise subprocess.CalledProcessError(p.returncode, cmd)
    return p

_MARK = "humanization_viz.py"          # 이 파일이 있는 폴더 = 가이드 루트

def _find_root():
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "가이드 루트를 못 찾았어요. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

GUIDE_ROOT = ROOT.resolve()
os.chdir(GUIDE_ROOT / CHAPTER)         # data/ 상대경로가 그대로 동작
sys.path.insert(0, str(GUIDE_ROOT))    # humanization_viz import
sys.path.insert(0, str(GUIDE_ROOT / CHAPTER))

_IMPORT_NAME = {"biopython": "Bio", "pyyaml": "yaml", "scikit-learn": "sklearn"}

def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p.replace("-", "_")))]
    if miss:
        print("설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))
        importlib.invalidate_caches()

_APT = APT_PKGS.split() if (APT_PKGS and IN_COLAB) else []   # 모아서 apt 를 한 번만 돌립니다
if PIP_PKGS:
    _ensure(PIP_PKGS)

def _ensure_pkg_resources():
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 합니다.
    if importlib.util.find_spec("pkg_resources") is None:
        _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
        importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    _APT.append("fonts-nanum")             # Colab 엔 한글 폰트가 없어 라벨이 □ 로 깨져요

if _APT:                                   # apt 인덱스 갱신은 한 번만 (ANARCI 는 hmmscan 이 필요해요)
    _run("apt-get -qq update", timeout=600, quiet=True)
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y " + " ".join(_APT), timeout=900, quiet=True)


# ── ① 내가 만든 결과(my_run) 우선 · ② 없으면 커밋된 레퍼런스(data/) ─────────────
MY  = pathlib.Path("my_run"); MY.mkdir(exist_ok=True)
REF = pathlib.Path("data")

def find_one(pattern, ref=REF, quiet=False):
    """산출물 하나 찾기 — 내가 만든 my_run/ 을 먼저 뒤지고, 없으면 커밋된 레퍼런스에서."""
    hits = sorted(MY.rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과]   {hits[0]}")
        return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"'{pattern}' 을 my_run/ 에서도 {ref}/ 에서도 찾지 못했어요."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def find_prev(chapter, pattern, ref=REF, quiet=False):
    """앞 챕터에서 직접 만든 산출물 → 없으면 이 챕터 data/ 레퍼런스."""
    hits = sorted((GUIDE_ROOT / chapter / "my_run").rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과 · {chapter}] {hits[0]}")
        return hits[0]
    return find_one(pattern, ref, quiet)

def read_fasta(path):
    out, name, buf = {}, None, []
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith(">"):
            if name: out[name] = "".join(buf)
            name, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if name: out[name] = "".join(buf)
    return out

def write_fasta(path, records):
    pathlib.Path(path).write_text("".join(f">{k}\n{v}\n" for k, v in records.items()))
    return pathlib.Path(path)

PARENTAL = read_fasta(REF / "parental.fasta")      # 가이드 전체를 관통하는 러닝 예제
VH = PARENTAL["parental_H"]
VL = PARENTAL["parental_L"]

print("가이드 루트 :", GUIDE_ROOT)
print("작업 폴더   :", pathlib.Path.cwd())
print(f"러닝 예제   : VH {len(VH)} aa · VL {len(VL)} aa (mouse hybridoma 가정)")

## 1) 직접 실행 — ANARCI numbering + germline (본문 4.1)

```bash
ANARCI -i parental.fasta --scheme imgt --assign_germline --use_species human --csv -o my_run/anarci_gl
```

`--assign_germline` 이 핵심이에요. 이게 있어야 "가장 가까운 사람 germline 과 몇 % 같은가" 가 나오고, 그 숫자가 humanization 전략을 정해요.

In [ ]:
t0 = time.time()
try:
    r = subprocess.run(["ANARCI", "-i", "data/parental.fasta", "--scheme", "imgt", "--csv",
                        "-o", str(MY / "anarci_gl"), "--assign_germline", "--use_species", "human"],
                       capture_output=True, text=True, timeout=300)
    rc, err = r.returncode, r.stderr
except FileNotFoundError as e:
    # ANARCI CLI 자체가 PATH 에 없음 (hmmscan 이 없으면 numbering 도 여기서 죽어요 — Ch.03)
    rc, err = 127, f"{e} — ANARCI/hmmscan 이 PATH 에 없어요"
except subprocess.TimeoutExpired:
    rc, err = 124, "300초를 넘겨 끊었어요"
el = time.time() - t0

gl_paths = sorted(MY.glob("anarci_gl_*.csv"))
if rc == 0 and gl_paths:
    print(f"ANARCI numbering+germline {el:.2f}초 · CSV {len(gl_paths)}개")
    for p in gl_paths:
        print("  →", p)
    print("판정 — 체인별 CSV(H · KL)가 생겼으면 numbering 과 germline 할당이 모두 끝난 거예요.")
else:
    print("ANARCI CLI 실패:", str(err).strip()[:300])
    print("→ hmmscan 이 PATH 에 있는지 확인하세요(Ch.03). 이 랩은 커밋된 실행 산출물로 이어집니다.")
    gl_paths = [pathlib.Path("data/anarci_imgt_H.csv"), pathlib.Path("data/anarci_imgt_KL.csv")]
    print("[레퍼런스]", " · ".join(str(p) for p in gl_paths))

## 2) 내 결과 확인 — germline 표와 종 판정 (본문 4.2)

진짜 정보는 germline 컬럼에 있어요. V identity 가 낮은 체인 = 사람 germline 과 멀다 = **humanization 여지가 크다**.

In [ ]:
import pandas as pd

def _cell(row, name):
    return row[name] if (name in row.index and pd.notna(row[name])) else None

def germline_table(paths):
    """ANARCI --csv 결과에서 체인·종·V/J germline 만 뽑는다. germline 컬럼이 없으면 (표, True)."""
    rows, missing = [], False
    for p in paths:
        for _, r in pd.read_csv(p).iterrows():
            vg, vi = _cell(r, "v_gene"), _cell(r, "v_identity")
            jg, ji = _cell(r, "j_gene"), _cell(r, "j_identity")
            if vg is None or vi is None:
                missing = True
            rows.append({"체인": _cell(r, "chain_type") or "?",
                         "hmm_species": _cell(r, "hmm_species") or "?",
                         "V gene": vg or "—", "V identity": float(vi) if vi is not None else float("nan"),
                         "J gene": jg or "—", "J identity": float(ji) if ji is not None else float("nan")})
    return pd.DataFrame(rows), missing

gl, no_germline = germline_table(gl_paths)
if no_germline:
    print("germline 컬럼이 없어요 — 1) 셀에 --assign_germline --use_species human 을 붙여 다시 실행하세요.")
    print("[레퍼런스] data/anarci_imgt_H.csv · data/anarci_imgt_KL.csv 로 이어갑니다")
    gl, _ = germline_table(["data/anarci_imgt_H.csv", "data/anarci_imgt_KL.csv"])
display(gl)

hv = gl.loc[gl["체인"] == "H", "V identity"]
lv = gl.loc[gl["체인"].isin(["K", "L"]), "V identity"]
if len(hv) and len(lv):
    h, l = float(hv.iloc[0]), float(lv.iloc[0])
    print(f"V identity — heavy {h:.0%} / light {l:.0%}. 낮은 쪽인 "
          f"{'heavy' if h < l else 'light'} 에 손댈 자리가 많아요. 노력의 무게중심을 여기에 둡니다.")

# HMM 종 순위 — 어느 종 프로파일이 이겼는지, 그리고 그 차이가 믿을 만한지
cur, hits = None, {}
for ln in pathlib.Path(find_one("anarci_hits.txt")).read_text().splitlines():
    f = ln.split()
    if not f:
        continue
    if f[0] == "NAME":
        cur = f[1]; hits[cur] = []
    elif cur and len(f) >= 5 and f[0][-2:] in ("_H", "_L", "_K"):
        hits[cur].append((f[0], float(f[2])))
for name, hs in hits.items():
    if len(hs) >= 2:
        gap = hs[0][1] - hs[1][1]
        verdict = "종 판정 근거로 쓸 만해요" if gap >= 10 else "사실상 동률이라 종 판정 근거로는 약해요"
        print(f"{name} HMM 1위 {hs[0][0]} {hs[0][1]} · 2위 {hs[1][0]} {hs[1][1]} (차 {gap:.1f}) — {verdict}")

## 3) 같은 서열, 다른 J 유전자 — 동점을 직접 재계산 (본문 4.2)

ANARCI 와 abnumber 는 같은 서열에 **서로 다른 J 유전자**를 답해요. 어느 쪽이 틀린 게 아니라 동점이에요.
말로 넘기지 말고 확인해요 — ANARCI 패키지 안에 IMGT 정렬된 사람 germline 세트가 들어 있으니,
**1) 에서 나온 IMGT 번호열을 그 세트 전체와 대조**하면 몇 개가 동률인지 그대로 보입니다.
(여기서 만드는 numbering 은 이 노트북 전체가 재사용해요 — 체인당 딱 한 번만 돌려요.)

In [ ]:
HAVE_ABNUMBER, chains, abn_j = _have("abnumber"), {}, "—"
if HAVE_ABNUMBER:
    try:
        from abnumber import Chain
        t0 = time.time()
        chains["H"] = Chain(VH, scheme="imgt", assign_germline=True)
        chains["L"] = Chain(VL, scheme="imgt", assign_germline=True)
        abn_j = chains["H"].j_gene
        print(f"abnumber numbering(H·L, germline 포함) {time.time()-t0:.2f}초")
    except Exception as e:
        HAVE_ABNUMBER = False
        print("abnumber 실패:", type(e).__name__, str(e)[:120])
        print("→ 5)·6) 은 커밋된 크로스워크/CDR 표로 이어져요.")

anarci_j = gl.loc[gl["체인"] == "H", "J gene"].iloc[0] if (gl["체인"] == "H").any() else "?"
anarci_v = gl.loc[gl["체인"] == "H", "V gene"].iloc[0] if (gl["체인"] == "H").any() else "?"
print("ANARCI J gene  :", anarci_j)
print("abnumber J gene:", abn_j)

# ── IMGT 번호열 하나를 뽑아 germline 세트 전체와 대조 ──────────────────────────
def imgt_row(paths, want="H"):
    for p in paths:
        d = pd.read_csv(p)
        sel = d[d["chain_type"] == want] if "chain_type" in d.columns else d
        if len(sel):
            cols = sorted([c for c in d.columns if str(c).isdigit()], key=int)
            return [str(sel.iloc[0][c]) for c in cols]
    return None

par_imgt = imgt_row(gl_paths, "H")
try:
    from anarci import germlines
    HAVE_GLSET = par_imgt is not None
except Exception as e:
    HAVE_GLSET = False
    print("anarci germline 세트를 못 읽었어요:", type(e).__name__, str(e)[:100])

def identity_scan(region):
    """parental H 의 IMGT 번호열 vs 사람 germline(V 또는 J) 전체 — gap 을 뺀 위치에서 % identity."""
    out = []
    for allele, gseq in germlines.all_germlines[region]["H"]["human"].items():
        pairs = [(a, b) for a, b in zip(gseq, par_imgt) if a != "-" and b != "-"]
        if not pairs:
            continue
        m = sum(a == b for a, b in pairs)
        out.append({"allele": allele, "일치": m, "정렬길이": len(pairs), "identity": m / len(pairs)})
    df = pd.DataFrame(out)
    return df.sort_values(["identity", "allele"], ascending=[False, True]).reset_index(drop=True)

def top_tie(df):
    return df[df["identity"] >= df["identity"].max() - 1e-9]

if HAVE_GLSET:
    tie_j = top_tie(identity_scan("J"))
    display(tie_j)
    r0 = tie_j.iloc[0]
    print(f"J 최고 동률 {len(tie_j)}개 — {int(r0['일치'])}/{int(r0['정렬길이'])} = {r0['identity']:.2%}")
    picked = [x for x in (anarci_j, abn_j) if isinstance(x, str) and x not in ("—", "?")]
    inside = [x for x in picked if x in set(tie_j["allele"])]
    print(f"두 도구가 고른 {len(picked)}개 중 {len(inside)}개가 이 동률 목록 안 — "
          "어느 쪽이 먼저 나오냐는 tie-break(참조 세트·순회 순서) 문제예요.")
print("판정 — J 절편은 14 잔기라 동점이 흔해요. backmutation 판단의 근거는 V 로 잡고, J 는 참고로만 봅니다.")

## 4) 레퍼런스 대조 — IgBLAST 로 V 유전자 교차검증 (본문 4.4)

ANARCI 의 germline 추정이 맞는지 두 번째 도구로 확인하면 더 든든해요. 진입장벽은 **germline DB 를 직접 마련해야** 한다는 점인데,
본문의 트릭이 여기서 통해요 — **ANARCI 패키지 안의 사람 germline 을 FASTA 로 뽑아** DB 재료로 씁니다(외부 다운로드 없음).

```bash
conda install -c bioconda igblast -y
makeigblastdb -in human_IGHV.fasta -dbtype prot -out db/human_gl_V
igblastp -query parental_H.fasta -germline_db_V db/human_gl_V -organism human -outfmt 7
```

`igblastp` 가 없어도 괜찮아요. 같은 germline 세트로 **% identity 를 직접 계산**해 같은 대조를 합니다.

In [ ]:
fa = None
if HAVE_GLSET:
    ref_v = germlines.all_germlines["V"]["H"]["human"]
    fa = MY / "human_IGHV.fasta"
    fa.write_text("".join(f">{a}\n{str(s).replace('-', '').replace('.', '')}\n" for a, s in ref_v.items()))
    print(f"human IGHV allele {len(ref_v)}개 → {fa}")

igb, mkdb = shutil.which("igblastp"), shutil.which("makeigblastdb")
if fa and igb and mkdb:
    db = MY / "db"; db.mkdir(exist_ok=True)
    q = MY / "parental_H.fasta"; q.write_text(f">parental_H\n{VH}\n")
    try:
        subprocess.run(f'makeigblastdb -in "{fa}" -dbtype prot -out "{db}/human_gl_V"',
                       shell=True, capture_output=True, text=True, timeout=300)
        r = subprocess.run(f'igblastp -query "{q}" -germline_db_V "{db}/human_gl_V" -organism human -outfmt 7',
                           shell=True, capture_output=True, text=True, timeout=300)
        rows = [ln.split("\t") for ln in r.stdout.splitlines() if ln.startswith("V\t")]
        for h in rows[:3]:
            print("  igblastp hit —", " ".join(h[1:5]), "(query · subject · %identity · 정렬길이)")
        if not rows:
            print("  igblastp 가 V 히트를 내지 않았어요:", (r.stderr or "").strip()[:200])
    except Exception as e:
        print("igblastp 실행 실패:", type(e).__name__, str(e)[:160])
else:
    print("igblastp/makeigblastdb 가 없어요 — conda install -c bioconda igblast 로 깔면 이 셀이 실제 BLAST 까지 돌려요.")

if HAVE_GLSET:
    tie_v = top_tie(identity_scan("V"))
    display(tie_v)
    r0 = tie_v.iloc[0]
    print(f"V 최고 동률 {len(tie_v)}개 — {int(r0['일치'])}/{int(r0['정렬길이'])} = {r0['identity']:.2%}")
    fams = sorted({str(a).split("-")[0].split("*")[0] for a in tie_v["allele"]})
    print("동률 allele 의 계열 —", ", ".join(fams), "· ANARCI 가 고른", anarci_v,
          "가 이 목록 안" if anarci_v in set(tie_v["allele"]) else "는 이 목록 밖")
    print(f"판정 — 계열이 {len(fams)}개면 정렬 방식이 다른 두 경로가 같은 곳을 가리킨 거예요. "
          "유전자 이름이 한 끗 달라도 subgroup 과 identity 가 같으면 결론은 하나예요.")
print("참고 — igblastp 는 단백질 모드라 J·junction 은 다루지 않아요. V gene 과 % identity 확인이 주 용도예요.")

## 5) 번호 체계가 두 개예요 — raw ↔ IMGT 크로스워크

뒤 챕터에서 도구별 mutation 표기가 섞여요.

| 체계 | 어디서 쓰나 | 예 |
|---|---|---|
| **raw 1-based** | Sapiens·AnthroAb 의 `predict_scores` 가 그대로 쓰는 서열 인덱스 | `I78T` = 입력 문자열의 78번째 잔기 |
| **IMGT** | ANARCI/abnumber numbering. gap·삽입이 있어 raw 와 어긋남 | `H86` = 같은 잔기의 IMGT 번호 |

Humatch 는 indel 을 만들 수 있어(우리 VL 은 1 잔기 짧아져요) **raw 인덱스 비교가 깨져요.** 그래서 도구 간 비교는 **반드시 IMGT 로 변환**해서 합니다.

In [ ]:
def raw2imgt(seq, ch):
    """raw 1-based 인덱스 → IMGT 라벨. numbering 밖(C-말단 꼬리)은 tail 로 표시."""
    assert seq.startswith(ch.seq), "numbering 영역이 서열 앞부분과 어긋나요 — 입력 서열을 확인하세요"
    m, pos_raw, last = {}, {}, None
    for i, (pos, aa) in enumerate(ch, start=1):
        m[i] = str(pos); pos_raw[pos] = i; last = str(pos)
    for k in range(1, len(ch.tail) + 1):
        m[len(ch.seq) + k] = f"{last}_tail{k}"
    return m, pos_raw

r2i, raw_of = {}, {}
if HAVE_ABNUMBER:
    for tag, seq in (("H", VH), ("L", VL)):
        r2i[tag], raw_of[tag] = raw2imgt(seq, chains[tag])
        (MY / f"raw2imgt_{tag}.json").write_text(
            json.dumps({str(k): v for k, v in r2i[tag].items()}, indent=1))
    print("→", MY / "raw2imgt_H.json", "·", MY / "raw2imgt_L.json")
else:
    for tag in ("H", "L"):
        p = find_one(f"raw2imgt_{tag}.json")
        r2i[tag] = {int(k): v for k, v in json.loads(pathlib.Path(p).read_text()).items()}

def _cross(chain, picks):        # 대괄호·중괄호 없이 "raw 78 → H86" 꼴로 읽히게
    return " · ".join(f"raw {k} → {r2i[chain][k]}" for k in picks if k in r2i[chain])
print("raw → IMGT (VH) —", _cross("H", (5, 12, 78, 115)))
print("raw → IMGT (VL) —", _cross("L", (31, 85, 109, 111)))
last_l = max(r2i["L"])
print(f"VL 마지막 잔기 raw {last_l} → {r2i['L'][last_l]} — IMGT 범위 밖이라 tail 로 라벨링돼요")
print("판정 — Sapiens 가 'I78T' 라고 하면 raw 78번이고, 같은 잔기의 IMGT 번호는",
      r2i["H"].get(78), "이에요. Ch.06 의 도구 간 합의는 이 변환을 거쳐야 성립해요.")

## 6) 직접 실행 — CDR 추출 + **보호 좌표 못 박기** (본문 4.3)

humanization 에서 가장 먼저 할 일은 "여기는 절대 안 건드린다"를 좌표로 고정하는 것이에요.
CDR 을 **raw 인덱스**(Sapiens/AnthroAb 가 쓰는 좌표)와 **IMGT 라벨** 두 가지로 저장하고, 본문 4.3 의 IMGT 규격창(**CDR1 27–38 · CDR2 56–65 · CDR3 105–117**) 안에 들어오는지까지 봅니다.
Ch.05 의 CDR 가드가 여기서 나온 파일을 그대로 씁니다.

In [ ]:
IMGT_WINDOW = {"CDR1": (27, 38), "CDR2": (56, 65), "CDR3": (105, 117)}   # 본문 4.3

def imgt_num(label):
    d = "".join(x for x in str(label) if x.isdigit())
    return int(d) if d else None

cdr_rows = []
if HAVE_ABNUMBER:
    for tag in ("H", "L"):
        ch = chains[tag]
        for name in ("CDR1", "CDR2", "CDR3"):
            if name not in ch.regions:
                continue
            reg = ch.regions[name]
            pos = list(reg.keys())
            raws = [raw_of[tag][p] for p in pos]
            cdr_rows.append({"chain": tag, "cdr": name, "sequence": "".join(reg.values()),
                             "raw_start": min(raws), "raw_end": max(raws),
                             "imgt": f"{pos[0]}..{pos[-1]}"})
else:
    print("[레퍼런스] data/cdr_table_imgt.csv 의 CDR 서열로 좌표를 복원해요")
    for _, r in pd.read_csv("data/cdr_table_imgt.csv").iterrows():
        tag, s = r["chain"], r["sequence"]
        st = (VH if tag == "H" else VL).find(s)
        if st < 0:
            print(f"  {tag}-{r['cdr']} 를 parental 에서 못 찾아 건너뜁니다(입력 서열이 레퍼런스와 다르면 생겨요)")
            continue
        cdr_rows.append({"chain": tag, "cdr": r["cdr"], "sequence": s,
                         "raw_start": st + 1, "raw_end": st + len(s),
                         "imgt": f"{r2i[tag].get(st + 1)}..{r2i[tag].get(st + len(s))}"})

guard = {"H": [], "L": []}
for row in cdr_rows:
    guard[row["chain"]] += list(range(row["raw_start"], row["raw_end"] + 1))
    lo, hi = IMGT_WINDOW[row["cdr"]]
    a, b = imgt_num(row["imgt"].split("..")[0]), imgt_num(row["imgt"].split("..")[-1])
    row["IMGT 규격창"] = f"{lo}–{hi}"
    row["규격 안"] = bool(a is not None and b is not None and lo <= a and b <= hi)

cdr = pd.DataFrame(cdr_rows)
if not len(cdr):
    print("CDR 좌표를 하나도 만들지 못했어요 — 3) 에서 abnumber 가 실패했고 레퍼런스 복원도 안 됐어요.")
else:
    display(cdr)
    if HAVE_ABNUMBER:
        cdr.to_csv(MY / "cdr_table_imgt.csv", index=False)
        (MY / "cdr_guard.json").write_text(json.dumps(guard, indent=1))
        print("→", MY / "cdr_table_imgt.csv", "·", MY / "cdr_guard.json")

    print("보호 좌표(raw 1-based) — VH", len(guard["H"]), "잔기 · VL", len(guard["L"]), "잔기")
    print(f"IMGT 규격창 안에 들어온 CDR {int(cdr['규격 안'].sum())}/{len(cdr)}개")
    h3 = cdr[(cdr["chain"] == "H") & (cdr["cdr"] == "CDR3")]
    if len(h3):
        print("판정 — CDR-H3 =", h3["sequence"].iloc[0],
              "· 항원 결합에 가장 결정적인 loop 예요. 여기 mutation 이 들어가면 빨간불이에요.")

## 7) 레퍼런스 대조 — 커밋된 실행 결과와 맞춰보기

`data/` 는 이 가이드를 만들 때 **실제로 돌려 나온** 산출물이에요. 내 결과와 한 글자씩 비교해요.

In [ ]:
ref_cdr   = pd.read_csv("data/cdr_table_imgt.csv")
ref_gl    = pd.read_csv("data/germline_assignment.csv")
ref_r2i_H = json.loads(pathlib.Path("data/raw2imgt_H.json").read_text())

got  = {(r.chain, r.cdr): r.sequence for r in cdr.itertuples()}
want = {(r.chain, r.cdr): r.sequence for r in ref_cdr.itertuples()}
common = sorted(set(got) & set(want))
print(f"CDR {len(common)}개 일치", all(got[k] == want[k] for k in common))
print("raw→IMGT(H) 일치", {str(k): v for k, v in r2i["H"].items()} == ref_r2i_H)
if not HAVE_ABNUMBER:
    print("(지금은 커밋된 레퍼런스로 좌표를 복원한 상태라 위 두 줄은 data/ 를 data/ 와 비교한 셈이에요.)")

rows = []
for _, r in ref_gl.iterrows():
    sel = gl[gl["체인"] == r["chain"]]
    col = f"{r['gene_type']} gene"
    mine_gene = str(sel[col].iloc[0]) if (len(sel) and col in gl.columns) else "—"
    mine_id   = float(sel[f"{r['gene_type']} identity"].iloc[0]) if (len(sel) and f"{r['gene_type']} identity" in gl.columns) else float("nan")
    rows.append({"체인": r["chain"], "종류": r["gene_type"],
                 "레퍼런스": f"{r['gene']} {r['identity_pct']}%",
                 "내 결과": f"{mine_gene} {mine_id:.1%}",
                 "같은 유전자": bool(mine_gene == r["gene"])})
chk = pd.DataFrame(rows)
display(chk)
print(f"같은 유전자 {int(chk['같은 유전자'].sum())}/{len(chk)}줄. "
      "ANARCI CSV 의 identity 는 소수 둘째 자리에서 반올림돼 있어 germline_assignment.csv 의 63.27% 가 0.63 으로 보여요.")
print("판정 — V 유전자가 두 줄 다 맞으면 입력 QC 통과예요. J 는 3) 에서 본 대로 동률이라 갈릴 수 있어요.")

## 이 랩에서 확인한 것

1. **germline 실측** — VH `IGHV1-69*06` **63.27%** / VL `IGLV1-40*01` **80.85%**. heavy 가 훨씬 비인간적이라 손댈 자리가 많아요.
2. **J 유전자는 동률** — 사람 germline 세트를 직접 훑으면 J 최고점이 **12/14 = 85.71%** 로 여러 allele 에 걸쳐 있어요. ANARCI 의 `IGHJ6*01` 과 abnumber 의 `IGHJ4*01` 이 모두 그 안이라 tie-break 만 갈립니다. 판단 근거는 V 로 잡아요.
3. **IgBLAST 교차검증** — ANARCI 패키지의 germline 을 FASTA 로 뽑아 DB 를 만들면 외부 다운로드 없이 재현돼요. top hit 은 `IGHV1-8*01`·`IGHV1-69*08` **63.27%** 로 ANARCI 와 **같은 IGHV1 계열·같은 identity** 예요. `igblastp` 는 J·junction 을 다루지 않아요.
4. **CDR 6개** — H `GYTFTDYV`/`IYPGSGTN`/`ARRGRYGLYAMDY`, L `SSDVGHKFP`/`KNL`/`QSYDSSLRVV`. 여섯 개 모두 IMGT 규격창(27–38 · 56–65 · 105–117) 안이에요.
5. **번호 체계 두 개**(raw 1-based ↔ IMGT)를 이어 붙인 `raw2imgt_*.json` 을 만들었어요. Ch.06 의 도구 간 합의 분석이 이 파일 위에서 돕니다.
6. **CDR 보호 좌표**(`cdr_guard.json`, VH 29 · VL 22 잔기)를 못 박았어요 — Ch.05 에서 이걸로 사고를 막아요.

다음 → **[Ch.05 — Sapiens](../05_humanize_sapiens/05_sapiens_lab.ipynb)**